In [ ]:
import sys
import subprocess
import importlib

def ensure_library_capability():
    try:
        import refractiveindex as ri
        # If these attributes are missing, the version is too old
        if not hasattr(ri, 'download_database'):
            print("Library version outdated. Upgrading...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "refractiveindex"])
            importlib.reload(ri)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "refractiveindex"])

ensure_library_capability()

import numpy as np
import matplotlib.pyplot as plt
import refractiveindex as ri
from refractiveindex import RefractiveIndexMaterial, refractiveindex as ri_sub

In [ ]:
# --- Material Database Setup ---
db_path = ri_sub._DEFAULT_DB_PATH
catalog_file = db_path / "catalog-nk.yml"

# Check if the database exists. If not, force download using the updated library
if not catalog_file.exists():
    print("Database missing. Attempting final download strategy...")
    try:
        # After the upgrade in Cell 1, this should now exist
        ri.download_database()
    except Exception as e:
        print(f"Standard download failed: {e}. Trying alternative...")
        # Direct CLI call as a last resort
        subprocess.run([sys.executable, "-m", "refractiveindex", "download"])

if not catalog_file.exists():
    raise FileNotFoundError(f"Could not download database to {db_path}. Check internet connection.")

# Load the catalog for the Abelès Transfer Matrix Method
catalog = ri_sub._load_catalog(db_path)

def find_best_material_path(name):
    """Searches the database for a matching material name."""
    for key in catalog.keys():
        if key[1].lower() == name.lower():
            return key
    return None

In [ ]:
def calculate_reflectance(layers, wavelength, mat_cache):
    """
    Implements the Abeles Transfer Matrix Method.
    M_layer = [[cos(phi), -i/n * sin(phi)], [-i*n*sin(phi), cos(phi)]]
    """
    M_total = np.eye(2, dtype=complex) # Start with an identity matrix (empty space)
    
    for mat, d in layers:
        # Resolve complex refractive index (n + ik)
        if isinstance(mat, str):
            obj = mat_cache[mat]
            n_val = obj.get_refractive_index(wavelength) # n_val is the real part (refraction)
            try: k_val = obj.get_extinction_coefficient(wavelength) # Try to get k (absorption). If not in DB, assume material is lossless (k=0)
            except: k_val = 0.0
            n_layer = complex(n_val + 1j * k_val)
        else:
            n_layer = complex(mat) # Fallback for manual numeric input from the slider

        phi = (2 * np.pi * n_layer * d) / wavelength # phi = the phase shift as light travels through the thickness 'd'
        
        # Characteristic matrix elements for one layer
        m11 = np.cos(phi)
        m12 = -1j/n_layer * np.sin(phi)
        m21 = -1j*n_layer * np.sin(phi)
        m22 = np.cos(phi)
        
        # Matrix multiplication chains the layers: M_total = M1 * M2 * ... * Mn
        M_layer = np.array([[m11, m12], [m21, m22]], dtype=complex)
        M_total = M_total @ M_layer

    # Assuming Air - n=1 - for both the incident medium and the substrate
    # B and C represent the normalized field amplitudes
    B = M_total[0,0] + M_total[0,1]
    C = M_total[1,0] + M_total[1,1]
    
    return np.abs((B - C) / (B + C))**2 # Square the magnitude of the reflection coefficient to get power reflectance

In [ ]:
def plot_sensitivity(layers, error_nm=2, iterations=10):
    """Runs a Monte Carlo simulation for manufacturing tolerances."""
    
    wl_range = np.linspace(400, 800, 400) # Visible light spectrum range in nanometers
    
    # Loading materials from the DB is slow, so a cache is needed
    cache = {}
    for mat, _ in layers:
        if isinstance(mat, str):
            path = find_best_material_path(mat)
            if path:
                cache[mat] = RefractiveIndexMaterial(shelf=path[0], book=path[1], page=path[2])
            else:
                print(f"Error: Material {mat} not found in database!")
                return

    plt.figure(figsize=(10, 6))
    
    # this loop simulates multiple 'failed' versions of the filter
    for _ in range(iterations):
        # Apply random noise to each layer's thickness to simulate deposition errors
        noisy_layers = []
        for mat, d in layers:
            noise = np.random.uniform(-error_nm, error_nm)
            noisy_layers.append((mat, d + noise))

        R_list = [calculate_reflectance(noisy_layers, wl, cache) * 100 for wl in wl_range] # Calculate reflectance across the whole spectrum for the noisy version

        plt.plot(wl_range, R_list, color='red', alpha=0.3) # alpha=0.3 makes lines semi-transparent so we can see the 'error cloud' density properly
        
    plt.title(f"Sensitivity Analysis: {layers[0][0]} (Error ±{error_nm}nm)")
    plt.xlabel("Wavelength (nm)")
    plt.ylabel("Reflectance (%)")
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
from interface import launch_interface
launch_interface(plot_sensitivity)